# Cacao Bean Image Classification with SVM + GLCM

This notebook automatically fetches a public Kaggle image dataset from code using `kagglehub`—you do **not** manually upload the dataset. It extracts HSV color statistics and GLCM texture features, trains an SVM, evaluates it, and downloads the trained model.

**Dataset source used here:** [Cocoa Ripeness Dataset TCS 01](https://www.kaggle.com/datasets/juanfelipeheredia/cocoa-ripeness-dataset-tcs-01). It is a real Kaggle cocoa image dataset, but its labels describe **ripeness/maturity**, not cut-bean commercial grades or mold/slaty defects. Therefore, this dataset is useful to test the code pipeline, but it is **not a perfect match** for your Grade 1 / Grade 2 / Reject target. For a valid quality-grade model, use an image dataset labelled with cut-test classes such as brown, partially purple, slaty, and moldy. Do not present ripeness-based results as validated commercial grading.

The dataset is retrieved automatically into the temporary Colab runtime by `kagglehub.dataset_download(...)`. Kaggle may require you to accept the dataset's terms or authenticate in Colab depending on its current access settings.

## 1. Install libraries

In [ ]:
!pip -q install kagglehub opencv-python-headless scikit-image scikit-learn joblib seaborn

## 2. Call the Kaggle dataset directly from code

In [ ]:
import kagglehub
from pathlib import Path

KAGGLE_DATASET = "juanfelipeheredia/cocoa-ripeness-dataset-tcs-01"

# Automatically fetches the public dataset into the Colab runtime.
# No manual download or upload is required.
dataset_root = Path(kagglehub.dataset_download(KAGGLE_DATASET))
print("Dataset available at:", dataset_root)
print("Top-level files/folders:", [p.name for p in dataset_root.iterdir()][:40])

## 3. Locate images and labels

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
image_paths = [p for p in dataset_root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
print("Number of images found:", len(image_paths))
for p in image_paths[:20]:
    print(p.relative_to(dataset_root))

if not image_paths:
    raise RuntimeError("No image files found. Inspect the dataset structure printed above.")

## 4. Extract HSV color + GLCM texture features

This feature extraction matches the requested SVM approach. GLCM features represent texture, while HSV statistics represent color.

In [ ]:
import cv2, re, json, joblib
import numpy as np
from skimage.feature import graycomatrix, graycoprops

def extract_features(image_path, size=(128, 128)):
    image = cv2.imread(str(image_path))
    if image is None:
        return None
    image = cv2.resize(image, size)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    color_features = []
    for channel in cv2.split(hsv):
        color_features.extend([float(channel.mean()), float(channel.std())])

    gray_q = (gray // 16).astype(np.uint8)
    glcm = graycomatrix(
        gray_q, distances=[1, 2],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=16, symmetric=True, normed=True
    )
    texture_features = []
    for prop in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]:
        texture_features.extend(graycoprops(glcm, prop).ravel().tolist())
    return np.asarray(color_features + texture_features, dtype=np.float32)


## 5. Read labels from folder names

This code expects class-labelled folders (for example, `immature/`, `mature/`, `overmature/`). If the dataset stores labels in a CSV instead, inspect its files and adapt this cell to read that CSV. The example maps maturity labels into the three website outputs only to demonstrate the pipeline; this mapping is **not a scientifically valid cacao quality grading rule**.

In [ ]:
def get_label(path, root):
    parts = list(path.relative_to(root).parts[:-1])
    ignored = {"train", "training", "test", "testing", "valid", "validation", "images", "image", "data"}
    for part in reversed(parts):
        if part.lower() not in ignored:
            return part
    return None

def map_to_website_grade(label):
    s = re.sub(r"[_-]+", " ", label.lower()).strip()
    # DEMONSTRATION ONLY: maturity is not equivalent to commercial cut-test quality.
    if any(x in s for x in ["mature", "ripened"]) and not any(x in s for x in ["immature", "overmature", "over mature"]):
        return "Grade 1 - Premium Export Quality"
    if any(x in s for x in ["immature", "under", "partially"]):
        return "Grade 2 - Standard Local Quality"
    if any(x in s for x in ["overmature", "over mature", "defect", "reject", "mold", "mould", "slaty"]):
        return "Reject / Substandard"
    return None

records = []
for p in image_paths:
    source_label = get_label(p, dataset_root)
    if source_label:
        target = map_to_website_grade(source_label)
        if target:
            records.append((p, source_label, target))

print("Mapped image count:", len(records))
print("Source labels found:", sorted(set(r[1] for r in records)))
print("Mapped classes:", sorted(set(r[2] for r in records)))

if len(records) < 10 or len(set(r[2] for r in records)) < 2:
    raise RuntimeError(
        "The dataset may store labels in a CSV or use different folder names. "
        "Inspect the file list above and modify get_label()/map_to_website_grade(). "
        "Do not continue with guessed labels."
    )

## 6. Train and evaluate the SVM

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

X_list, y_list = [], []
for path, source_label, grade in records:
    features = extract_features(path)
    if features is not None:
        X_list.append(features)
        y_list.append(grade)

X = np.vstack(X_list)
y = np.asarray(y_list)
classes, counts = np.unique(y, return_counts=True)
print("Feature matrix shape:", X.shape)
print("Class counts:", dict(zip(classes, counts)))

if len(classes) < 2 or counts.min() < 2:
    raise ValueError("Need at least two classes and at least two images per class.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=10, gamma="scale", probability=True,
                class_weight="balanced", random_state=42))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Test accuracy:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, zero_division=0))


In [ ]:
labels = sorted(set(y_test) | set(pred))
cm = confusion_matrix(y_test, pred, labels=labels)
plt.figure(figsize=(9, 6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.title("SVM Confusion Matrix")
plt.xticks(rotation=25, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 7. Save and download the trained model

In [ ]:
MODEL_PATH = "/content/cacao_bean_svm_glcm.joblib"
bundle = {
    "model": model,
    "class_names": sorted(set(y)),
    "feature_type": "HSV channel mean/std + GLCM texture properties",
    "image_size": [128, 128],
    "dataset": KAGGLE_DATASET,
    "important_warning": (
        "This selected Kaggle dataset describes cocoa ripeness, not commercial cut-test "
        "quality. The example grade mapping is for pipeline demonstration only."
    )
}
joblib.dump(bundle, MODEL_PATH)
print("Model saved:", MODEL_PATH)

from google.colab import files
files.download(MODEL_PATH)


## 8. Predict a new image

In [ ]:
def predict_cacao_grade(image_path, model_bundle=bundle):
    features = extract_features(image_path)
    if features is None:
        raise ValueError(f"Could not read image: {image_path}")
    fitted_model = model_bundle["model"]
    probabilities = fitted_model.predict_proba([features])[0]
    idx = int(np.argmax(probabilities))
    return {
        "predicted_grade": str(fitted_model.classes_[idx]),
        "confidence": float(probabilities[idx]),
        "probabilities": {
            str(label): float(prob)
            for label, prob in zip(fitted_model.classes_, probabilities)
        }
    }

# Example after uploading a test image for inference:
# print(predict_cacao_grade("/content/test_cut_bean.jpg"))
